# Notebook 04: Deep Candidate Vetting Report

Load an existing candidate from the summary CSV and produce deeper diagnostic plots and a final vetting report.

Assumes Notebook 02 or 03 has already been run and the candidate's data is cached.

In [ ]:
import sys, logging
sys.path.insert(0, '../src')
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

import pandas as pd
import matplotlib
matplotlib.use('inline')
import matplotlib.pyplot as plt
%matplotlib inline

from pathlib import Path
from tesshunter import download, preprocessing, bls_search, plotting, vetting, catalog_crossmatch, reporting
from tesshunter.config import CANDIDATES_DIR, PLOTS_DIR
from tesshunter.utils import phase_fold
import numpy as np

## Choose a candidate from the summary

In [ ]:
SUMMARY_CSV = CANDIDATES_DIR / 'candidate_summary.csv'

if not SUMMARY_CSV.exists():
    raise FileNotFoundError(f"{SUMMARY_CSV} not found. Run notebook 03 first.")

summary = pd.read_csv(SUMMARY_CSV)
print(f"Loaded {len(summary)} candidate(s):")
display(summary[['tic_id','period_days','depth_ppt','snr','verdict']])

In [ ]:
# ── Pick a candidate row ────────────────────────────────────────────────────
ROW_INDEX = 0    # 0 = highest-SNR candidate; change as needed
# ────────────────────────────────────────────────────────────────────────────

row = summary.iloc[ROW_INDEX]
TIC_ID = int(row['tic_id'])
TARGET_LABEL = f"TIC {TIC_ID}"
PERIOD = float(row['period_days'])
DEPTH_PPT = float(row['depth_ppt'])

print(f"Selected: TIC {TIC_ID}  |  Period: {PERIOD:.4f} d  |  Depth: {DEPTH_PPT:.3f} ppt")
print(f"Previous verdict: {row['verdict']}")

## Re-download and preprocess (uses local cache)

In [ ]:
lcc = download.search_and_download(TIC_ID)
sectors = download.sectors_available(lcc)
lc_raw = download.stitch_collection(lcc)
lc_clean, prep_record = preprocessing.preprocess(lc_raw, do_flatten=True)
print(f"Ready: {len(lc_clean)} cadences, {len(sectors)} sector(s)")

## Re-run BLS at finer resolution around best period

In [ ]:
# Wide search first to confirm
_, bls_result, candidates = bls_search.run_bls(lc_clean, period_min=0.5, period_max=20.0, top_n=5)
best = candidates.iloc[0]
print(f"BLS best period: {best['period_days']:.6f} d  (depth: {best['depth_ppt']:.3f} ppt)")

## Odd/even transit depth test

In [ ]:
from tesshunter.vetting import check_odd_even

flagged, delta_sigma, note = check_odd_even(
    lc_clean,
    period_days=best['period_days'],
    epoch_btjd=best['epoch_btjd'],
    duration_days=best['duration_days'],
)
print(f"Odd/even check: {note}")
print(f"Flagged: {flagged}")

## Secondary eclipse check

In [ ]:
from tesshunter.vetting import check_secondary_eclipse

sec_flagged, sec_depth_ppt, sec_note = check_secondary_eclipse(
    lc_clean,
    period_days=best['period_days'],
    epoch_btjd=best['epoch_btjd'],
    duration_days=best['duration_days'],
)
print(sec_note)
print(f"Secondary flagged: {sec_flagged}")

## Full phase-folded light curve (full phase range, looking for secondary)

In [ ]:
time = lc_clean.time.value
flux = lc_clean.flux.value
phase = phase_fold(time, best['period_days'], best['epoch_btjd'])
sort = np.argsort(phase)

fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(phase[sort], flux[sort], s=1, alpha=0.3, c='steelblue')
half_dur = best['duration_days'] / best['period_days'] / 2
ax.axvspan(-half_dur, half_dur, alpha=0.15, color='gold', label='Primary transit')
ax.axvspan(0.5 - half_dur, 0.5 + half_dur, alpha=0.10, color='red', label='Phase 0.5 (secondary window)')
ax.axvspan(-0.5 - half_dur, -0.5 + half_dur, alpha=0.10, color='red')
ax.set_xlabel('Phase')
ax.set_ylabel('Normalized Flux')
ax.set_title(f'{TARGET_LABEL} — Full Phase, P={best["period_days"]:.4f} d')
ax.legend()
plt.tight_layout()
plt.show()

## Full vetting and final report

In [ ]:
xmatch = catalog_crossmatch.crossmatch(TIC_ID)
stellar_radius = (
    xmatch['tic_metadata'].get('stellar_radius_rsun') if xmatch['tic_metadata'] else None
)

vet = vetting.run_vetting(
    lc_clean,
    period_days=best['period_days'],
    epoch_btjd=best['epoch_btjd'],
    duration_days=best['duration_days'],
    depth_ppt=best['depth_ppt'],
    n_sectors=len(sectors),
    stellar_radius_rsun=stellar_radius,
    is_known_toi=xmatch['is_known_toi'],
)

print(f"FINAL VERDICT: {vet.verdict}")
if vet.estimated_radius_rearth:
    print(f"Estimated radius: {vet.estimated_radius_rearth:.2f} R_Earth")

In [ ]:
plot_paths = plotting.all_diagnostic_plots(lc_raw, lc_clean, bls_result, candidates, TARGET_LABEL)

report_path = reporting.generate_report(
    target_id=TARGET_LABEL,
    tic_id=TIC_ID,
    sectors=sectors,
    prep_record=prep_record,
    candidates_df=candidates,
    vetting=vet,
    crossmatch_summary=xmatch['summary_str'],
    plot_paths=plot_paths,
)
print(f"Deep vetting report: {report_path}")